# Qwen-VL 入门教程

Qwen-VL 是阿里巴巴推出的视觉语言模型，通过 **Visual Resampler** 实现高效的视觉-语言对齐。

## 学习目标

- 理解 Visual Resampler 的工作原理
- 掌握 Qwen-VL 的架构设计
- 学会使用 Qwen-VL 进行多模态任务

## 目录

1. [什么是 Qwen-VL](#1-什么是-qwen-vl)
2. [Visual Resampler](#2-visual-resampler)
3. [模型架构](#3-模型架构)
4. [代码实践](#4-代码实践)
5. [总结](#5-总结)

## 1. 什么是 Qwen-VL

### 设计目标

Qwen-VL 旨在解决视觉语言模型的两个关键问题：
1. **视觉 token 过多**: ViT 输出大量 patch token，增加计算负担
2. **特征对齐**: 视觉和语言特征空间不一致

### 解决方案

使用 **Visual Resampler** (类似 BLIP-2 的 Q-Former)：
- 将可变数量的视觉 token 压缩为固定数量的 query token
- 通过交叉注意力学习视觉-语言对齐

In [ ]:
# 环境准备
import sys
sys.path.insert(0, '../src')

import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'使用设备: {device}')

torch.manual_seed(42)

## 2. Visual Resampler

### 工作原理

```
视觉编码器输出: [B, N, D]  (N 个 patch token)
        ↓
Visual Resampler (交叉注意力)
        ↓
压缩后特征: [B, M, D]  (M 个 query token, M << N)
```

### 关键组件

1. **可学习 Query**: 固定数量的可学习向量
2. **交叉注意力**: Query 从视觉特征中提取信息
3. **位置编码**: 保留空间位置信息

In [ ]:
# 可视化 Visual Resampler
fig, ax = plt.subplots(figsize=(12, 6))
ax.set_xlim(0, 12)
ax.set_ylim(0, 8)
ax.axis('off')
ax.set_title('Visual Resampler 工作流程', fontsize=14, fontweight='bold')

# 视觉编码器输出
ax.add_patch(plt.Rectangle((0.5, 5), 3, 2), color='lightblue', ec='black')
ax.text(2, 6, '视觉特征\n[B, 256, D]', ha='center', va='center', fontsize=10)

# Query tokens
ax.add_patch(plt.Rectangle((0.5, 1), 3, 2), color='lightyellow', ec='black')
ax.text(2, 2, '可学习 Query\n[B, 64, D]', ha='center', va='center', fontsize=10)

# 交叉注意力
ax.add_patch(plt.Rectangle((5, 3), 2.5, 2), color='lightgreen', ec='black')
ax.text(6.25, 4, '交叉注意力', ha='center', va='center', fontsize=10)

# 输出
ax.add_patch(plt.Rectangle((9, 3), 2.5, 2), color='lightcoral', ec='black')
ax.text(10.25, 4, '压缩特征\n[B, 64, D]', ha='center', va='center', fontsize=10)

# 箭头
ax.annotate('', xy=(5, 4.5), xytext=(3.5, 6), arrowprops=dict(arrowstyle='->', color='gray'))
ax.annotate('', xy=(5, 3.5), xytext=(3.5, 2), arrowprops=dict(arrowstyle='->', color='gray'))
ax.annotate('', xy=(9, 4), xytext=(7.5, 4), arrowprops=dict(arrowstyle='->', color='gray'))

ax.text(4.2, 5.5, 'K, V', fontsize=9, color='blue')
ax.text(4.2, 2.8, 'Q', fontsize=9, color='blue')

plt.tight_layout()
plt.show()

## 3. 模型架构

In [ ]:
from qwen_vl import QwenVLConfig, QwenVL

# 创建小型模型
config = QwenVLConfig(
    image_size=224,
    patch_size=14,
    vision_layers=4,
    vision_width=256,
    vision_heads=4,
    num_query_tokens=64,  # 压缩后的 token 数
    hidden_size=512,
    num_layers=4,
    num_heads=8,
    num_kv_heads=8,
    intermediate_size=1024,
    vocab_size=32000
)

model = QwenVL(config).to(device)
print(f'模型参数量: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# 查看配置
print('=== Qwen-VL 配置 ===')
print(f'图像大小: {config.image_size}')
print(f'Patch 大小: {config.patch_size}')
print(f'视觉 token 数: {(config.image_size // config.patch_size) ** 2}')
print(f'压缩后 token 数: {config.num_query_tokens}')
print(f'压缩比: {(config.image_size // config.patch_size) ** 2 / config.num_query_tokens:.1f}x')

## 4. 代码实践

### 4.1 视觉特征提取与压缩

In [ ]:
# 测试视觉特征提取
images = torch.randn(2, 3, 224, 224).to(device)

with torch.no_grad():
    # 经过 Visual Resampler 压缩后的特征
    vision_features = model.get_vision_features(images)

print(f'压缩后视觉特征: {vision_features.shape}')
print(f'每张图像 {vision_features.shape[1]} 个 token')

### 4.2 多模态前向传播

In [ ]:
# 完整前向传播
batch_size = 2
input_ids = torch.randint(0, 32000, (batch_size, 128)).to(device)
images = torch.randn(batch_size, 3, 224, 224).to(device)

with torch.no_grad():
    outputs = model(input_ids, images)

print(f'输出 logits: {outputs["logits"].shape}')

## 5. 总结

### Qwen-VL 的特点

| 特性 | 说明 |
|------|------|
| Visual Resampler | 压缩视觉 token，降低计算量 |
| 可学习 Query | 自适应提取关键视觉信息 |
| 交叉注意力 | 实现视觉-语言对齐 |

### 与其他模型对比

| 模型 | 视觉 token 处理 |
|------|----------------|
| LLaVA | 直接投影，保留所有 token |
| BLIP-2 | Q-Former 压缩 |
| Qwen-VL | Visual Resampler 压缩 |
| CogVLM | Visual Expert 处理 |

### 适用场景

- 高分辨率图像理解
- 长文本生成
- 多图像输入